# Module 04.14 — Configuration (`config`)

> Part of the **Elasticsearch Snapshot Training** course.
> Run `00_setup.ipynb` first to start the Docker stack and load sample data.


In [ ]:
import sys, json, time
sys.path.insert(0, '..')
from helpers import *

es = get_client()

---
## 4.14 Configuration (`config`)

The `config` saved object is Kibana's **per-version, per-space settings store**. It
captures the values you set in **Stack Management → Advanced Settings**: the default Data
View, date and number formats, timezone, Dark Mode preference, and feature tour completion
flags, among others.

The object ID is the Kibana version string (e.g. `9.3.0`), which means each Kibana version
creates its own `config` object. When you upgrade Kibana, the new version reads the old
config object and migrates settings forward, then writes a new config object for the new
version. This is why after a major-version upgrade you may find multiple `config` objects
in the Saved Objects UI.

The most operationally significant attribute is `defaultIndex` — the ID of the Data View
that Kibana opens by default in Discover. If that Data View ID does not exist (e.g. after
a partial restore or a cluster migration), Discover will prompt the user to select a Data
View manually. Everything else in `config` is cosmetic and will degrade gracefully if
missing.

Because `config` is namespace-scoped, each Kibana Space has its own config object.
Restoring the full Kibana feature state brings back the config for every space.

In [ ]:
heading('4.14 Config saved object')

configs = find_saved_objects('config')
for c in configs:
    console.print(f"  id={c['id']}")
    pp(c['attributes'], 'config attributes')
    info('Key attributes:')
    info('  defaultIndex       — default data view ID (set when you star a data view)')
    info('  dateFormat         — display format for dates in Kibana')
    info('  theme:darkMode     — UI theme preference')
    info('  timepicker:*       — saved time picker defaults')
    info('  The config id is the Kibana version string (e.g. 9.3.0)')
    break

In [ ]:
heading('4.14 Config — modify a setting via saved objects API')

# The config object ID is the Kibana version string
configs = find_saved_objects('config')
config_id = configs[0]['id']  # e.g. '9.3.0'
pp(configs[0]['attributes'], 'Current config attributes')

# Set timepicker default to 7 days so we can verify it survives a restore
import requests as req
req.put(
    f'{KIBANA_HOST}/api/saved_objects/config/{config_id}',
    auth=('elastic', KIBANA_PASSWORD),
    headers={'kbn-xsrf': 'true', 'Content-Type': 'application/json'},
    json={'attributes': {'timepicker:timeDefaults': '{"from":"now-7d","to":"now"}'}},
).raise_for_status()
success('Set timepicker default to last 7 days')

In [ ]:
heading('Config — snapshot → change setting → restore → verify')

import requests as req

# Snapshot with timepicker=7d captured
snap_delete_restore_cycle('snap-config-test', 'Config')

# Change the setting to something different
req.put(
    f'{KIBANA_HOST}/api/saved_objects/config/{config_id}',
    auth=('elastic', KIBANA_PASSWORD),
    headers={'kbn-xsrf': 'true', 'Content-Type': 'application/json'},
    json={'attributes': {'timepicker:timeDefaults': '{"from":"now-1d","to":"now"}'}},
).raise_for_status()
warn('Changed timepicker to last 1 day — about to restore back to 7 days')

current = kibana_get(f'/api/saved_objects/config/{config_id}')
info(f'timepicker before restore: {current["attributes"].get("timepicker:timeDefaults")}')

restore_kibana_state(es, SNAP_REPO, 'snap-config-test')
time.sleep(3)

after = kibana_get(f'/api/saved_objects/config/{config_id}')
tp_after = after['attributes'].get('timepicker:timeDefaults', '')
info(f'timepicker after restore: {tp_after}')
assert 'now-7d' in str(tp_after), f'Expected 7d setting to be restored, got: {tp_after}'
success('Config setting restored correctly — timepicker is back to last 7 days')